<a href="https://colab.research.google.com/github/hamza-24-ai/ca_research/blob/main/CA_TASK_01.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Matrix Multiplication on Cpu

In [ ]:
import torch, time, statistics, csv

def sync(device):
    if device.type == "cuda":
        torch.cuda.synchronize()

def bench_matmul(device, n, reps=10, warmup=3):
    a = torch.randn(n, n, device=device)
    b = torch.randn(n, n, device=device)
    for _ in range(warmup):          # warm-up runs, not timed
        c = a @ b
    sync(device)
    times = []
    for _ in range(reps):
        t0 = time.perf_counter()
        c = a @ b
        sync(device)                 # wait for device before stopping timer
        times.append(time.perf_counter() - t0)
    flops = 2 * n**3
    med = statistics.median(times)
    return {
        "device": str(device), "n": n, "reps": reps,
        "mean_s": statistics.mean(times),
        "median_s": med,
        "std_s": statistics.stdev(times),
        "gflops_median": flops / med / 1e9,
    }

device = torch.device("cpu")
print("PyTorch", torch.__version__, "| threads:", torch.get_num_threads())
rows = [bench_matmul(device, n) for n in [256, 512, 1024, 1536, 2048]]
for r in rows:
    print(r)

with open("matmul_cpu.csv", "w", newline="") as f:
    w = csv.DictWriter(f, fieldnames=rows[0].keys())
    w.writeheader(); w.writerows(rows)
print("Saved matmul_cpu.csv")

PyTorch 2.14.1+cpu | threads: 1
{'device': 'cpu', 'n': 256, 'reps': 10, 'mean_s': 0.0011246811999171768, 'median_s': 0.0010533604997817747, 'std_s': 0.00022106461784087163, 'gflops_median': 31.854651856559542}
{'device': 'cpu', 'n': 512, 'reps': 10, 'mean_s': 0.008863102499935849, 'median_s': 0.009272266499920079, 'std_s': 0.0013968623250414317, 'gflops_median': 28.95036030320243}
{'device': 'cpu', 'n': 1024, 'reps': 10, 'mean_s': 0.06077273040000364, 'median_s': 0.05800572649991409, 'std_s': 0.005898216743188292, 'gflops_median': 37.02192486121488}
{'device': 'cpu', 'n': 1536, 'reps': 10, 'mean_s': 0.11409899350005617, 'median_s': 0.11244952850029222, 'std_s': 0.005722774700283777, 'gflops_median': 64.45342553820637}
{'device': 'cpu', 'n': 2048, 'reps': 10, 'mean_s': 0.26641833899998346, 'median_s': 0.26565789250003036, 'std_s': 0.006267327419502991, 'gflops_median': 64.66914655659266}
Saved matmul_cpu.csv


# Memory Bandwidth on Cpu

In [ ]:
import torch, time, statistics, csv

def bench_copy(n_elems, reps=10, warmup=3):
    src = torch.randn(n_elems)          # float32, 4 bytes per element
    dst = torch.empty_like(src)
    for _ in range(warmup):
        dst.copy_(src)
    times = []
    for _ in range(reps):
        t0 = time.perf_counter()
        dst.copy_(src)
        times.append(time.perf_counter() - t0)
    med = statistics.median(times)
    bytes_moved = 2 * n_elems * 4       # read src + write dst
    return {
        "op": "copy", "n_elems": n_elems, "size_MB": n_elems * 4 / 1e6,
        "reps": reps, "median_s": med,
        "std_s": statistics.stdev(times),
        "GBps_median": bytes_moved / med / 1e9,
    }

rows = [bench_copy(n) for n in [10_000_000, 50_000_000, 200_000_000]]
for r in rows:
    print(r)
with open("bandwidth_cpu.csv", "w", newline="") as f:
    w = csv.DictWriter(f, fieldnames=rows[0].keys())
    w.writeheader(); w.writerows(rows)
print("Saved bandwidth_cpu.csv")

{'op': 'copy', 'n_elems': 10000000, 'size_MB': 40.0, 'reps': 10, 'median_s': 0.009646715000144468, 'std_s': 0.0007871754049963622, 'GBps_median': 8.292978490481156}
{'op': 'copy', 'n_elems': 50000000, 'size_MB': 200.0, 'reps': 10, 'median_s': 0.04285845300000801, 'std_s': 0.002747302816053581, 'GBps_median': 9.333048022053555}
{'op': 'copy', 'n_elems': 200000000, 'size_MB': 800.0, 'reps': 10, 'median_s': 0.1591916715001389, 'std_s': 0.005705128790830598, 'GBps_median': 10.050777059644126}
Saved bandwidth_cpu.csv


# CPU Precision on FP32,BF16,FP16

In [ ]:
import torch, time, statistics, csv
torch.manual_seed(0)

def timed(fn, reps=10, warmup=3):
    for _ in range(warmup):
        fn()
    ts = []
    for _ in range(reps):
        t0 = time.perf_counter()
        fn()
        ts.append(time.perf_counter() - t0)
    return ts

n = 1024
a = torch.randn(n, n)
b = torch.randn(n, n)
ref = a @ b                          # FP32 reference result

rows = []
for name, dt in [("fp32", torch.float32), ("bf16", torch.bfloat16), ("fp16", torch.float16)]:
    try:
        ad, bd = a.to(dt), b.to(dt)
        ts = timed(lambda: ad @ bd)
        out = (ad @ bd).float()
        rel_err = ((out - ref).norm() / ref.norm()).item()
        med = statistics.median(ts)
        rows.append({"precision": name, "n": n, "median_s": med,
                     "std_s": statistics.stdev(ts),
                     "gflops_median": 2 * n**3 / med / 1e9,
                     "rel_error_vs_fp32": rel_err, "status": "ok"})
    except Exception as e:
        rows.append({"precision": name, "n": n, "status": f"not supported: {type(e).__name__}"})

rows.append({"precision": "int8", "n": n, "status": "not performed on CPU in this test"})
rows.append({"precision": "transfer", "n": n, "status": "not performed (CPU has no separate device)"})

for r in rows:
    print(r)
with open("precision_cpu.csv", "w", newline="") as f:
    w = csv.DictWriter(f, fieldnames=sorted({k for r in rows for k in r}))
    w.writeheader(); w.writerows(rows)
print("Saved precision_cpu.csv")

{'precision': 'fp32', 'n': 1024, 'median_s': 0.03395370499993078, 'std_s': 0.010892607783584866, 'gflops_median': 63.24740254426956, 'rel_error_vs_fp32': 0.0, 'status': 'ok'}
{'precision': 'bf16', 'n': 1024, 'median_s': 4.1313857684999675, 'std_s': 0.14731211266901642, 'gflops_median': 0.5197974162504105, 'rel_error_vs_fp32': 0.0028795425314456224, 'status': 'ok'}
{'precision': 'fp16', 'n': 1024, 'median_s': 4.074948043499944, 'std_s': 0.4203137102340685, 'gflops_median': 0.5269965715085637, 'rel_error_vs_fp32': 0.0003594137670006603, 'status': 'ok'}
{'precision': 'int8', 'n': 1024, 'status': 'not performed on CPU in this test'}
{'precision': 'transfer', 'n': 1024, 'status': 'not performed (CPU has no separate device)'}
Saved precision_cpu.csv


# **Move to GPU**

# Matrix Multiplication

In [ ]:
import torch, time, statistics, csv

def sync(device):
    if device.type == "cuda":
        torch.cuda.synchronize()

def bench_matmul(device, n, reps=10, warmup=3):
    a = torch.randn(n, n, device=device)
    b = torch.randn(n, n, device=device)
    for _ in range(warmup):          # warm-up runs, not timed
        c = a @ b
    sync(device)
    times = []
    for _ in range(reps):
        t0 = time.perf_counter()
        c = a @ b
        sync(device)                 # wait for device before stopping timer
        times.append(time.perf_counter() - t0)
    flops = 2 * n**3
    med = statistics.median(times)
    return {
        "device": str(device), "n": n, "reps": reps,
        "mean_s": statistics.mean(times),
        "median_s": med,
        "std_s": statistics.stdev(times),
        "gflops_median": flops / med / 1e9,
    }

device = torch.device("cuda")
print("PyTorch", torch.__version__, "| threads:", torch.get_num_threads())
rows = [bench_matmul(device, n) for n in [256, 512, 1024, 1536, 2048]]
for r in rows:
    print(r)

with open("matmul_gpu.csv", "w", newline="") as f:
    w = csv.DictWriter(f, fieldnames=rows[0].keys())
    w.writeheader(); w.writerows(rows)
print("Saved matmul_gpu.csv")

PyTorch 2.14.1+cu130 | threads: 1
{'device': 'cuda', 'n': 256, 'reps': 10, 'mean_s': 6.021949998853415e-05, 'median_s': 5.656600001202605e-05, 'std_s': 9.34073016877836e-06, 'gflops_median': 593.1908212153282}
{'device': 'cuda', 'n': 512, 'reps': 10, 'mean_s': 0.00012313469999298833, 'median_s': 0.00011914049997585607, 'std_s': 1.2156397195917458e-05, 'gflops_median': 2253.0999622663885}
{'device': 'cuda', 'n': 1024, 'reps': 10, 'mean_s': 0.0006496055999946293, 'median_s': 0.0006436859999894295, 'std_s': 2.1858997901797622e-05, 'gflops_median': 3336.2286084135203}
{'device': 'cuda', 'n': 1536, 'reps': 10, 'mean_s': 0.002065956699993876, 'median_s': 0.002062618499991231, 'std_s': 8.794418682222824e-06, 'gflops_median': 3513.862263928503}
{'device': 'cuda', 'n': 2048, 'reps': 10, 'mean_s': 0.004781728099999327, 'median_s': 0.004777461499969604, 'std_s': 2.014268655217706e-05, 'gflops_median': 3596.024621885347}
Saved matmul_gpu.csv


# GPU Bandwidth

In [ ]:
import torch, time, statistics, csv

def bench_copy(n_elems, reps=10, warmup=3):
    src = torch.randn(n_elems, device="cuda")   # float32, 4 bytes each
    dst = torch.empty_like(src)
    for _ in range(warmup):
        dst.copy_(src)
    torch.cuda.synchronize()
    ts = []
    for _ in range(reps):
        t0 = time.perf_counter()
        dst.copy_(src)
        torch.cuda.synchronize()        # wait for GPU before stopping timer
        ts.append(time.perf_counter() - t0)
    med = statistics.median(ts)
    return {"device": "cuda", "op": "copy", "n_elems": n_elems,
            "size_MB": n_elems * 4 / 1e6, "reps": reps,
            "median_s": med, "std_s": statistics.stdev(ts),
            "GBps_median": 2 * n_elems * 4 / med / 1e9}

rows = [bench_copy(n) for n in [10_000_000, 50_000_000, 200_000_000]]
for r in rows:
    print(r)
with open("bandwidth_gpu.csv", "w", newline="") as f:
    w = csv.DictWriter(f, fieldnames=rows[0].keys())
    w.writeheader(); w.writerows(rows)
print("Saved bandwidth_gpu.csv")

{'device': 'cuda', 'op': 'copy', 'n_elems': 10000000, 'size_MB': 40.0, 'reps': 10, 'median_s': 0.0003478049999898758, 'std_s': 3.3240253933459308e-06, 'GBps_median': 230.01394460208655}
{'device': 'cuda', 'op': 'copy', 'n_elems': 50000000, 'size_MB': 200.0, 'reps': 10, 'median_s': 0.001665874499963138, 'std_s': 1.1972849287527683e-05, 'GBps_median': 240.11412624951703}
{'device': 'cuda', 'op': 'copy', 'n_elems': 200000000, 'size_MB': 800.0, 'reps': 10, 'median_s': 0.00663604649997751, 'std_s': 1.876187189615924e-05, 'GBps_median': 241.1074123735303}
Saved bandwidth_gpu.csv


# Precision in GPU

In [ ]:
import torch, time, statistics, csv
torch.manual_seed(0)

def timed(fn, reps=10, warmup=3):
    for _ in range(warmup):
        fn()
    torch.cuda.synchronize()
    ts = []
    for _ in range(reps):
        t0 = time.perf_counter()
        fn()
        torch.cuda.synchronize()
        ts.append(time.perf_counter() - t0)
    return ts

n = 1024
a = torch.randn(n, n, device="cuda")
b = torch.randn(n, n, device="cuda")
ref = a @ b                                    # FP32 reference

rows = []
for name, dt in [("fp32", torch.float32), ("fp16", torch.float16), ("bf16", torch.bfloat16)]:
    ad, bd = a.to(dt), b.to(dt)
    ts = timed(lambda: ad @ bd)
    out = (ad @ bd).float()
    rel_err = ((out - ref).norm() / ref.norm()).item()
    med = statistics.median(ts)
    rows.append({"device": "cuda", "precision": name, "n": n, "median_s": med,
                 "std_s": statistics.stdev(ts),
                 "gflops_median": 2 * n**3 / med / 1e9,
                 "rel_error_vs_fp32": rel_err, "status": "ok"})

# INT8 matmul on GPU (integer inputs, int32 output)
try:
    ai = torch.randint(-128, 127, (n, n), dtype=torch.int8, device="cuda")
    bi = torch.randint(-128, 127, (n, n), dtype=torch.int8, device="cuda")
    ts = timed(lambda: torch._int_mm(ai, bi))
    med = statistics.median(ts)
    rows.append({"device": "cuda", "precision": "int8", "n": n, "median_s": med,
                 "std_s": statistics.stdev(ts),
                 "gflops_median": 2 * n**3 / med / 1e9,
                 "rel_error_vs_fp32": "not comparable (integer output)", "status": "ok"})
except Exception as e:
    rows.append({"device": "cuda", "precision": "int8", "n": n,
                 "status": f"not supported: {type(e).__name__}: {e}"})

rows.append({"device": "cuda", "precision": "transfer", "n": n,
             "status": "see transfer test (next step)"})

for r in rows:
    print(r)
with open("precision_gpu.csv", "w", newline="") as f:
    w = csv.DictWriter(f, fieldnames=sorted({k for r in rows for k in r}))
    w.writeheader(); w.writerows(rows)
print("Saved precision_gpu.csv")

{'device': 'cuda', 'precision': 'fp32', 'n': 1024, 'median_s': 0.000877391499955138, 'std_s': 3.914147372807574e-05, 'gflops_median': 2447.5774475930107, 'rel_error_vs_fp32': 0.0, 'status': 'ok'}
{'device': 'cuda', 'precision': 'fp16', 'n': 1024, 'median_s': 0.00016465750002225832, 'std_s': 6.984665603975471e-06, 'gflops_median': 13042.124699510829, 'rel_error_vs_fp32': 0.0003597258764784783, 'status': 'ok'}
{'device': 'cuda', 'precision': 'bf16', 'n': 1024, 'median_s': 0.0015114655000161292, 'std_s': 1.8241046831897247e-05, 'gflops_median': 1420.795676763435, 'rel_error_vs_fp32': 0.0028806175105273724, 'status': 'ok'}
{'device': 'cuda', 'precision': 'int8', 'n': 1024, 'median_s': 0.0003857519999996839, 'std_s': 2.6171445398853737e-06, 'gflops_median': 5567.005869060328, 'rel_error_vs_fp32': 'not comparable (integer output)', 'status': 'ok'}
{'device': 'cuda', 'precision': 'transfer', 'n': 1024, 'status': 'see transfer test (next step)'}
Saved precision_gpu.csv


# GPU Transfer test

In [ ]:
import torch, time, statistics, csv

def bench_transfer(n_elems, reps=10, warmup=3):
    host = torch.randn(n_elems).pin_memory()     # pinned CPU memory
    for _ in range(warmup):
        host.to("cuda", non_blocking=False)
    torch.cuda.synchronize()
    ts = []
    for _ in range(reps):
        t0 = time.perf_counter()
        d = host.to("cuda", non_blocking=False)
        torch.cuda.synchronize()
        ts.append(time.perf_counter() - t0)
    med = statistics.median(ts)
    size_bytes = n_elems * 4
    return {"device": "cuda", "direction": "host_to_device", "n_elems": n_elems,
            "size_MB": size_bytes / 1e6, "reps": reps, "median_s": med,
            "std_s": statistics.stdev(ts),
            "GBps_median": size_bytes / med / 1e9}

rows = [bench_transfer(n) for n in [10_000_000, 50_000_000, 200_000_000]]
for r in rows:
    print(r)
with open("transfer_gpu.csv", "w", newline="") as f:
    w = csv.DictWriter(f, fieldnames=rows[0].keys())
    w.writeheader(); w.writerows(rows)
print("Saved transfer_gpu.csv")

{'device': 'cuda', 'direction': 'host_to_device', 'n_elems': 10000000, 'size_MB': 40.0, 'reps': 10, 'median_s': 0.0033228014999622246, 'std_s': 1.4934262269218768e-05, 'GBps_median': 12.03803477290315}
{'device': 'cuda', 'direction': 'host_to_device', 'n_elems': 50000000, 'size_MB': 200.0, 'reps': 10, 'median_s': 0.0164047769999911, 'std_s': 0.0007128747126197684, 'GBps_median': 12.191570784540898}
{'device': 'cuda', 'direction': 'host_to_device', 'n_elems': 200000000, 'size_MB': 800.0, 'reps': 10, 'median_s': 0.06494030950005936, 'std_s': 3.096100149086605e-05, 'GBps_median': 12.319005039532016}
Saved transfer_gpu.csv


# **Move to TPU**

# Matrix Multiplication

In [ ]:
import torch, torch_tpu, time, statistics, csv

device = torch.device("tpu")
print("PyTorch", torch.__version__, "| device:", device)

def sync_tpu(x):
    # forces the TPU to finish before the timer stops
    return x.sum().item()

def bench_matmul(n, reps=10, warmup=3):
    a = torch.randn(n, n, device=device)
    b = torch.randn(n, n, device=device)
    for _ in range(warmup):                 # warm-up (includes compilation), not timed
        c = a @ b
        sync_tpu(c)
    ts = []
    for _ in range(reps):
        t0 = time.perf_counter()
        c = a @ b
        sync_tpu(c)                         # wait for TPU before stopping timer
        ts.append(time.perf_counter() - t0)
    med = statistics.median(ts)
    return {"device": "tpu", "n": n, "reps": reps,
            "mean_s": statistics.mean(ts), "median_s": med,
            "std_s": statistics.stdev(ts),
            "gflops_median": 2 * n**3 / med / 1e9}

rows = [bench_matmul(n) for n in [256, 512, 1024, 1536, 2048]]
for r in rows:
    print(r)
with open("matmul_tpu.csv", "w", newline="") as f:
    w = csv.DictWriter(f, fieldnames=rows[0].keys())
    w.writeheader(); w.writerows(rows)
print("Saved matmul_tpu.csv")

Successfully renamed PrivateUse1 backend to 'tpu'. Device: device(type='tpu')
Registered Python module for 'tpu'.
Device type: tpu, Device index: default
Initializing TPU distributed runtime


PyTorch 2.14.1+cpu | device: tpu
{'device': 'tpu', 'n': 256, 'reps': 10, 'mean_s': 0.0005097676999866963, 'median_s': 0.0005029440001180774, 'std_s': 3.4468659565280745e-05, 'gflops_median': 66.71603994107166}
{'device': 'tpu', 'n': 512, 'reps': 10, 'mean_s': 0.00041824599993560695, 'median_s': 0.00041236999982174893, 'std_s': 3.430870635175516e-05, 'gflops_median': 650.957771215253}
{'device': 'tpu', 'n': 1024, 'reps': 10, 'mean_s': 0.0004443780000656261, 'median_s': 0.00044724500003212597, 'std_s': 2.2242817994337953e-05, 'gflops_median': 4801.582237578385}
{'device': 'tpu', 'n': 1536, 'reps': 10, 'mean_s': 0.0004820410000320408, 'median_s': 0.0004827199998089782, 'std_s': 1.0514972879066334e-05, 'gflops_median': 15014.412733816873}
{'device': 'tpu', 'n': 2048, 'reps': 10, 'mean_s': 0.0005642460000217398, 'median_s': 0.0005619799999294628, 'std_s': 1.5321235185436363e-05, 'gflops_median': 30570.250162205662}
Saved matmul_tpu.csv


# TPU Memory Bandwidth

In [ ]:
import torch, torch_tpu, time, statistics, csv

device = torch.device("tpu")

def sync_tpu(x):
    return x.sum().item()

def bench_copy(n_elems, reps=10, warmup=3):
    src = torch.randn(n_elems, device=device)
    dst = torch.empty_like(src)
    for _ in range(warmup):
        dst.copy_(src)
        sync_tpu(dst)
    ts = []
    for _ in range(reps):
        t0 = time.perf_counter()
        dst.copy_(src)
        sync_tpu(dst)
        ts.append(time.perf_counter() - t0)
    med = statistics.median(ts)
    return {"device": "tpu", "op": "copy", "n_elems": n_elems,
            "size_MB": n_elems * 4 / 1e6, "reps": reps,
            "median_s": med, "std_s": statistics.stdev(ts),
            "GBps_median": 2 * n_elems * 4 / med / 1e9}

rows = [bench_copy(n) for n in [10_000_000, 50_000_000, 200_000_000]]
for r in rows:
    print(r)
with open("bandwidth_tpu.csv", "w", newline="") as f:
    w = csv.DictWriter(f, fieldnames=rows[0].keys())
    w.writeheader(); w.writerows(rows)
print("Saved bandwidth_tpu.csv")

{'device': 'tpu', 'op': 'copy', 'n_elems': 10000000, 'size_MB': 40.0, 'reps': 10, 'median_s': 0.0004502799997680995, 'std_s': 4.095497069559227e-05, 'GBps_median': 177.66722937106047}
{'device': 'tpu', 'op': 'copy', 'n_elems': 50000000, 'size_MB': 200.0, 'reps': 10, 'median_s': 0.0007117699997252203, 'std_s': 1.661235697821694e-05, 'GBps_median': 561.9792912800772}
{'device': 'tpu', 'op': 'copy', 'n_elems': 200000000, 'size_MB': 800.0, 'reps': 10, 'median_s': 0.0015237249999699998, 'std_s': 1.2010385280587585e-05, 'GBps_median': 1050.0582454389748}
Saved bandwidth_tpu.csv


# Memory Bandwidth TPU V2

In [ ]:
import torch, torch_tpu, time, statistics, csv

device = torch.device("tpu")

def sync_tpu(x):
    return x.sum().item()

def bench_rw(n_elems, reps=10, warmup=3):
    src = torch.randn(n_elems, device=device)
    for _ in range(warmup):
        y = src * 2.0 + 1.0                 # read src, write y (not skippable)
        sync_tpu(y)
    ts = []
    for _ in range(reps):
        t0 = time.perf_counter()
        y = src * 2.0 + 1.0
        sync_tpu(y)
        ts.append(time.perf_counter() - t0)
    med = statistics.median(ts)
    return {"device": "tpu", "op": "scale_add (read+write)", "n_elems": n_elems,
            "size_MB": n_elems * 4 / 1e6, "reps": reps,
            "median_s": med, "std_s": statistics.stdev(ts),
            "GBps_median": 2 * n_elems * 4 / med / 1e9}

rows = [bench_rw(n) for n in [10_000_000, 50_000_000, 200_000_000]]
for r in rows:
    print(r)
with open("bandwidth_tpu_v2.csv", "w", newline="") as f:
    w = csv.DictWriter(f, fieldnames=rows[0].keys())
    w.writeheader(); w.writerows(rows)
print("Saved bandwidth_tpu_v2.csv")

{'device': 'tpu', 'op': 'scale_add (read+write)', 'n_elems': 10000000, 'size_MB': 40.0, 'reps': 10, 'median_s': 0.0007028700001683319, 'std_s': 1.836645572396835e-05, 'GBps_median': 113.81905612821805}
{'device': 'tpu', 'op': 'scale_add (read+write)', 'n_elems': 50000000, 'size_MB': 200.0, 'reps': 10, 'median_s': 0.001956185500148422, 'std_s': 2.516296831010665e-05, 'GBps_median': 204.47958538167816}
{'device': 'tpu', 'op': 'scale_add (read+write)', 'n_elems': 200000000, 'size_MB': 800.0, 'reps': 10, 'median_s': 0.006500904999938939, 'std_s': 0.00011657916884758597, 'GBps_median': 246.1195787378878}
Saved bandwidth_tpu_v2.csv


# TPU Precision Test

In [ ]:
import torch, torch_tpu, time, statistics, csv
torch.manual_seed(0)
device = torch.device("tpu")

def sync_tpu(x):
    return x.sum().item()

def timed(fn, reps=10, warmup=3):
    for _ in range(warmup):
        sync_tpu(fn())
    ts = []
    for _ in range(reps):
        t0 = time.perf_counter()
        sync_tpu(fn())
        ts.append(time.perf_counter() - t0)
    return ts

n = 1024
a = torch.randn(n, n, device=device)
b = torch.randn(n, n, device=device)
ref = (a @ b).float().cpu()                      # FP32 reference (on TPU, CPU copy for error)

rows = []
for name, dt in [("fp32", torch.float32), ("bf16", torch.bfloat16), ("fp16", torch.float16)]:
    try:
        ad, bd = a.to(dt), b.to(dt)
        ts = timed(lambda: ad @ bd)
        out = (ad @ bd).float().cpu()
        rel_err = ((out - ref).norm() / ref.norm()).item()
        med = statistics.median(ts)
        rows.append({"device": "tpu", "precision": name, "n": n, "median_s": med,
                     "std_s": statistics.stdev(ts),
                     "gflops_median": 2 * n**3 / med / 1e9,
                     "rel_error_vs_fp32": rel_err, "status": "ok"})
    except Exception as e:
        rows.append({"device": "tpu", "precision": name, "n": n,
                     "status": f"not supported: {type(e).__name__}: {e}"})

try:
    ai = torch.randint(-128, 127, (n, n), dtype=torch.int8, device=device)
    bi = torch.randint(-128, 127, (n, n), dtype=torch.int8, device=device)
    ts = timed(lambda: torch._int_mm(ai, bi))
    med = statistics.median(ts)
    rows.append({"device": "tpu", "precision": "int8", "n": n, "median_s": med,
                 "std_s": statistics.stdev(ts),
                 "gflops_median": 2 * n**3 / med / 1e9,
                 "rel_error_vs_fp32": "not comparable (integer output)", "status": "ok"})
except Exception as e:
    rows.append({"device": "tpu", "precision": "int8", "n": n,
                 "status": f"not supported: {type(e).__name__}: {e}"})

rows.append({"device": "tpu", "precision": "transfer", "n": n,
             "status": "transfer test: see note (host-to-device)"})

for r in rows:
    print(r)
with open("precision_tpu.csv", "w", newline="") as f:
    w = csv.DictWriter(f, fieldnames=sorted({k for r in rows for k in r}))
    w.writeheader(); w.writerows(rows)
print("Saved precision_tpu.csv")

{'device': 'tpu', 'precision': 'fp32', 'n': 1024, 'median_s': 0.0003922250002688088, 'std_s': 2.281900257024475e-05, 'gflops_median': 5475.131994462965, 'rel_error_vs_fp32': 0.0, 'status': 'ok'}
{'device': 'tpu', 'precision': 'bf16', 'n': 1024, 'median_s': 0.0003822100002253137, 'std_s': 1.940302445525936e-05, 'gflops_median': 5618.59618203096, 'rel_error_vs_fp32': 0.0016627284931018949, 'status': 'ok'}
{'device': 'tpu', 'precision': 'fp16', 'n': 1024, 'median_s': 0.00042252499997630366, 'std_s': 5.2932786327618545e-05, 'gflops_median': 5082.500794320897, 'rel_error_vs_fp32': 0.0020437368657439947, 'status': 'ok'}
{'device': 'tpu', 'precision': 'int8', 'n': 1024, 'status': "not supported: RuntimeError: operator 'aten::_int_mm' is not implemented for TPU. Please file a feature request"}
{'device': 'tpu', 'precision': 'transfer', 'n': 1024, 'status': 'transfer test: see note (host-to-device)'}
Saved precision_tpu.csv


# TPU Transfer Test

In [ ]:
import torch, torch_tpu, time, statistics, csv

device = torch.device("tpu")

def bench_transfer(n_elems, reps=10, warmup=3):
    host = torch.randn(n_elems)
    for _ in range(warmup):
        d = host.to(device)
        d.sum().item()                      # wait for transfer to finish
    ts = []
    for _ in range(reps):
        t0 = time.perf_counter()
        d = host.to(device)
        d.sum().item()                      # wait for transfer to finish
        ts.append(time.perf_counter() - t0)
    med = statistics.median(ts)
    size_bytes = n_elems * 4
    return {"device": "tpu", "direction": "host_to_device", "n_elems": n_elems,
            "size_MB": size_bytes / 1e6, "reps": reps, "median_s": med,
            "std_s": statistics.stdev(ts),
            "GBps_median": size_bytes / med / 1e9}

rows = [bench_transfer(n) for n in [10_000_000, 50_000_000, 200_000_000]]
for r in rows:
    print(r)
with open("transfer_tpu.csv", "w", newline="") as f:
    w = csv.DictWriter(f, fieldnames=rows[0].keys())
    w.writeheader(); w.writerows(rows)
print("Saved transfer_tpu.csv")

{'device': 'tpu', 'direction': 'host_to_device', 'n_elems': 10000000, 'size_MB': 40.0, 'reps': 10, 'median_s': 0.0513070514998617, 'std_s': 0.004116646165357015, 'GBps_median': 0.7796199319718815}
{'device': 'tpu', 'direction': 'host_to_device', 'n_elems': 50000000, 'size_MB': 200.0, 'reps': 10, 'median_s': 0.131030710500454, 'std_s': 0.0013543054294188962, 'GBps_median': 1.5263597307541656}
{'device': 'tpu', 'direction': 'host_to_device', 'n_elems': 200000000, 'size_MB': 800.0, 'reps': 10, 'median_s': 0.25061499749972427, 'std_s': 0.0024419226536749002, 'GBps_median': 3.1921473494453587}
Saved transfer_tpu.csv
